In [2]:
%load_ext autoreload
%autoreload 2

import sys; sys.path.append("..")      # lets the notebook import our project code from src/
import numpy as np
import pandas as pd

from src.evaluation import load_train, poisson_deviance, PROCESSED, N_FOLDS, EXPOSURE_EDGES
from src.features import LADDER, DRIVAGE_EDGES, DRIVAGE_LABELS, VEHAGE_EDGES, VEHAGE_LABELS

train = load_train()
glm = pd.read_csv(PROCESSED / "oof_glm.csv")
gbm = pd.read_csv(PROCESSED / "oof_gbm.csv")
assert (glm["IDpol"].to_numpy() == train["IDpol"].to_numpy()).all() and (gbm["IDpol"].to_numpy() == train["IDpol"].to_numpy()).all()

y = train["ClaimNb"].to_numpy(float)
mu_glm, mu_gbm = glm["pred_claims"].to_numpy(), gbm["pred_claims"].to_numpy()

def unit_deviance(y, mu):
    """Per-policy deviance contribution; its mean is the poisson_deviance used everywhere."""
    term = np.zeros_like(y); pos = y > 0
    term[pos] = y[pos] * np.log(y[pos] / mu[pos])
    return 2 * (term - (y - mu))

diff = unit_deviance(y, mu_glm) - unit_deviance(y, mu_gbm)          # positive = boosting is better on that policy
masks = [(train["Fold"] == f).to_numpy() for f in range(N_FOLDS)]
gap_folds = np.array([poisson_deviance(y[m], mu_glm[m]) - poisson_deviance(y[m], mu_gbm[m]) for m in masks])
print("mean per-policy difference:", round(diff.mean(), 5), "| mean of per-fold gaps:", round(gap_folds.mean(), 5))   # both about 0.0083
assert abs(diff.mean() - gap_folds.mean()) < 5e-4                   # consistency check

bm = train["BonusMalus"]
segments = {
    "DrivAge band": pd.cut(train["DrivAge"], DRIVAGE_EDGES, labels=DRIVAGE_LABELS),
    "VehAge band": pd.cut(train["VehAge"], VEHAGE_EDGES, labels=VEHAGE_LABELS),
    "BonusMalus class": pd.Series(np.select([bm == 50, bm == 100, ~bm.isin(LADDER)], ["50", "100", "off ladder"],
                                            default="ladder 51-95"), index=train.index),
    "Exposure band": pd.cut(train["Exposure"], EXPOSURE_EDGES),
    "VehBrand": train["VehBrand"],
}

def where_gbm_wins(key):
    g = pd.DataFrame({"key": key, "diff": diff, "y": y, "glm": mu_glm, "gbm": mu_gbm}).groupby("key", observed=True).agg(
        policies=("y", "size"), claims=("y", "sum"), glm=("glm", "sum"), gbm=("gbm", "sum"), diff_sum=("diff", "sum"))
    g["share_of_policies"] = g["policies"] / g["policies"].sum()
    g["share_of_gap"] = g["diff_sum"] / g["diff_sum"].sum()          # share of the total GLM-minus-boosting deviance
    g["actual/GLM"] = g["claims"] / g["glm"]                          # calibration inside the segment
    g["actual/GBM"] = g["claims"] / g["gbm"]
    return g[["policies", "claims", "share_of_policies", "share_of_gap", "actual/GLM", "actual/GBM"]].round(3)

for name, key in segments.items():
    print(f"=== {name} ===")
    print(where_gbm_wins(key).to_string(), "\n")

# Two-way view for a suspected interaction: share of the gap (in %) by driver age band x bonus-malus class
pair = pd.DataFrame({"a": segments["DrivAge band"], "b": segments["BonusMalus class"], "diff": diff}) \
         .groupby(["a", "b"], observed=True)["diff"].sum().unstack() / diff.sum()
print((pair * 100).round(1).to_string())

mean per-policy difference: 0.00829 | mean of per-fold gaps: 0.00829
=== DrivAge band ===
        policies  claims  share_of_policies  share_of_gap  actual/GLM  actual/GBM
key                                                                              
18-21       9142   804.0              0.017         0.020       1.001       0.986
22-25      22066  1477.0              0.041         0.049       0.992       0.987
26-30      52390  2464.0              0.097         0.097       1.001       1.005
31-40     135715  6023.0              0.251         0.257       1.001       1.001
41-50     131719  7270.0              0.243         0.244       1.000       1.003
51-60     107984  5826.0              0.199         0.188       0.999       0.995
61-70      50919  2853.0              0.094         0.084       1.004       1.010
71-100     31497  2115.0              0.058         0.060       1.002       1.002 

=== VehAge band ===
       policies  claims  share_of_policies  share_of_gap  actual/GLM

In [3]:
AVG_GAP = diff.mean()                                    # average GLM-minus-boosting deviance per policy (about 0.0083)

bm = train["BonusMalus"]
bm_class = pd.Series(np.select([bm == 50, bm == 100, ~bm.isin(LADDER)], ["50", "100", "off ladder"],
                               default="ladder 51-95"), index=train.index)

def segment_report(mask, key):
    """Gap breakdown restricted to the policies in `mask`, by the levels of `key`."""
    g = pd.DataFrame({"key": key[mask].to_numpy(), "diff": diff[mask], "y": y[mask],
                      "glm": mu_glm[mask], "gbm": mu_gbm[mask]}).groupby("key", observed=True).agg(
        policies=("y", "size"), claims=("y", "sum"), glm=("glm", "sum"), gbm=("gbm", "sum"), diff_sum=("diff", "sum"))
    g["share_of_policies"] = g["policies"] / g["policies"].sum()
    g["share_of_gap"] = g["diff_sum"] / g["diff_sum"].sum()                  # share of the SUBSET's gap
    g["gap_vs_avg"] = (g["diff_sum"] / g["policies"]) / AVG_GAP              # gap per policy relative to the overall average
    g["actual/GLM"] = g["claims"] / g["glm"]
    g["actual/GBM"] = g["claims"] / g["gbm"]
    return g[["policies", "share_of_policies", "share_of_gap", "gap_vs_avg", "actual/GLM", "actual/GBM"]].round(3)

age0 = (train["VehAge"] == 0).to_numpy()
inside_age0 = {
    "DrivAge band": pd.cut(train["DrivAge"], DRIVAGE_EDGES, labels=DRIVAGE_LABELS),
    "BonusMalus class": bm_class,
    "Brand": pd.Series(np.where(train["VehBrand"] == "B12", "B12", "other"), index=train.index),
    "Exposure band": pd.cut(train["Exposure"], EXPOSURE_EDGES),
    "Density band": pd.cut(train["Density"], [0, 50, 200, 500, 1500, 5000, 30000]),
    "VehGas": train["VehGas"],
}
print("age-0 vehicles: share of policies", round(age0.mean(), 3),
      "| share of the total gap", round(diff[age0].sum() / diff.sum(), 3), "\n")
for name, key in inside_age0.items():
    print(f"=== inside age-0 vehicles: {name} ===")
    print(segment_report(age0, key).to_string(), "\n")

# Separate B12 from age 0: the four combinations
cells = pd.DataFrame({"vehicle": np.where(age0, "age 0", "age 1+"), "brand": inside_age0["Brand"].to_numpy(),
                      "diff": diff, "y": y, "glm": mu_glm, "gbm": mu_gbm})
t = cells.groupby(["vehicle", "brand"]).agg(policies=("y", "size"), claims=("y", "sum"),
                                            glm=("glm", "sum"), gbm=("gbm", "sum"), gap=("diff", "sum"))
t["share_of_policies"] = t["policies"] / t["policies"].sum()
t["share_of_gap"] = t["gap"] / t["gap"].sum()
t["gap_vs_avg"] = (t["gap"] / t["policies"]) / AVG_GAP
t["actual/GLM"] = t["claims"] / t["glm"]
t["actual/GBM"] = t["claims"] / t["gbm"]
print(t[["policies", "share_of_policies", "share_of_gap", "gap_vs_avg", "actual/GLM", "actual/GBM"]].round(3).to_string())

age-0 vehicles: share of policies 0.085 | share of the total gap 0.62 

=== inside age-0 vehicles: DrivAge band ===
        policies  share_of_policies  share_of_gap  gap_vs_avg  actual/GLM  actual/GBM
key                                                                                  
18-21        319              0.007         0.004       4.465       0.517       0.933
22-25       1653              0.036         0.030       6.105       0.746       0.921
26-30       3588              0.078         0.090       8.394       0.978       0.972
31-40      10395              0.226         0.277       8.962       1.212       1.065
41-50      11389              0.248         0.260       7.672       1.041       0.987
51-60      10319              0.224         0.208       6.773       0.989       1.020
61-70       5345              0.116         0.077       4.845       0.919       1.004
71-100      2984              0.065         0.053       5.990       0.831       0.950 

=== inside age-0 vehic